<a href="https://colab.research.google.com/github/wdittaya/data-analytics-workshop/blob/main/LLM_Workshop_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Your Own "Chat-with-Doc" AI

## **Step 1: Setting up the Google Colab Environment**

In [ ]:
# Install all necessary dependencies
!pip install -qU langchain langchain-community langchain-huggingface langchain-google-genai chromadb sentence-transformers pypdf langchain-text-splitters

## **Step 2: Upload a Sample Document**


In [ ]:
from google.colab import files

print("Please upload your PDF file.")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'User uploaded file "{filename}"')


## **Step 3: Document Processing (Load and Split)**


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the document using the filename from the upload step
try:
    loader = PyPDFLoader(filename)
    documents = loader.load()

    # 2. Split the document into chunks
    ## TODO: You may have to modify the chunk_size and overlap
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
    chunks = text_splitter.split_documents(documents)

    print(f"Successfully split the document into {len(chunks)} chunks.")
    print("Sample content from the first chunk:")
    print(chunks[0].page_content[:500])
except NameError:
    print("Error: 'filename' not defined. Please run Step 2 (Upload) first.")

## **Step 4: Create Embeddings and the Vector Database**

To find the right chunk of text later, we need to convert our text into numbers (Embeddings) and store them in a database (ChromaDB). We will use a free, open-source embedding model from HuggingFace that runs entirely inside your Colab notebook.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Download a free, fast embedding model
## TODO: Change this to a model that support your language
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Store the document chunks into a ChromaDB vector store
vector_db = Chroma.from_documents(chunks, embeddings_model)

# 3. Create a retriever (a tool that searches the database)
retriever = vector_db.as_retriever(search_kwargs={"k": 2}) # k=2 means it will fetch the top 2 most relevant chunks

print("Vector database created and populated!")


## **Step 5: Set up the LLM and the RAG Workflow**


In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# 1. Set up a modern decoder-only model (Qwen2.5 is excellent for RAG)
## TODO: Change this to a model that support your language
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

# Standard text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1, ## low temperature for deterministic behavior
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

# 2. Define the Prompt using Instruction format
system_prompt = (
    "You are a helpful assistant. Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. Keep the answer concise.\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Build the RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("RAG Workflow updated with Qwen2.5 and langchain_classic imports.")

## **Step 6: Query your document using RAG**

In [ ]:
def ask_question(query):
    print(f"Question: {query}")
    response = rag_chain.invoke({"input": query})
    print(f"Answer: {response['answer']}\n")

In [ ]:
query = "What is the main idea of the document?"
ask_question(query)